# ⚡ Lily Fast Video Studio — Kaggle Edition

**Goal:** the shortest practical path from **prompt / reference image → MP4** on a free Kaggle GPU.

This notebook uses **LTX-Video 0.9.8 2B Distilled** and deliberately removes the slow extras:
- no prompt-enhancer LLM
- no spatial-upscaler second pass
- no ComfyUI
- the model loads **once** and stays in GPU memory
- native distilled timestep schedule
- FP16 automatically on T4 / P100-class GPUs
- optional first-frame image conditioning
- phone-friendly Gradio interface

### Before running
In Kaggle, open **Settings → Accelerator → GPU** and turn **Internet ON**.

Then choose **Run All**. The first startup downloads the model and text encoder. After that, each Generate click reuses the already-loaded pipeline.

In [ ]:
# 1) INSTALL — run once per Kaggle session
import os, subprocess, sys, pathlib

ROOT = pathlib.Path("/kaggle/working/LTX-Video")
os.environ["HF_HOME"] = "/kaggle/working/hf-cache"
os.environ["TRANSFORMERS_CACHE"] = "/kaggle/working/hf-cache"

if not ROOT.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/Lightricks/LTX-Video.git", str(ROOT)],
        check=True,
    )

os.chdir(ROOT)

# Install the official inference dependencies + our tiny UI.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", ".[inference]",
     "gradio>=5,<7", "bitsandbytes>=0.45", "imageio-ffmpeg"],
    check=True,
)

print("✅ Installed.")


In [ ]:
# 2) GPU CHECK + SPEED SETTINGS
import torch, os, gc, platform

assert torch.cuda.is_available(), "❌ No GPU found. In Kaggle: Settings → Accelerator → GPU."

GPU_NAME = torch.cuda.get_device_name(0)
CC = torch.cuda.get_device_capability(0)
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3

# T4/P100 do not have native BF16 tensor-core support. FP16 is much better there.
DTYPE = torch.bfloat16 if CC[0] >= 8 else torch.float16

torch.backends.cuda.matmul.allow_tf32 = CC[0] >= 8
torch.backends.cudnn.allow_tf32 = CC[0] >= 8

print(f"GPU: {GPU_NAME}")
print(f"VRAM: {VRAM_GB:.1f} GB")
print(f"Compute capability: {CC[0]}.{CC[1]}")
print(f"Generation dtype: {DTYPE}")
print("✅ GPU ready.")


In [ ]:
# 3) DOWNLOAD + LOAD THE MODEL ONCE
#
# We construct the official LTX pipeline ourselves so T4 can use FP16
# instead of the stock BF16 config. The text encoder is loaded in 8-bit
# when possible to leave much more VRAM for video activations.

import json, os, gc, torch
from pathlib import Path
from safetensors import safe_open
from huggingface_hub import hf_hub_download
from transformers import T5EncoderModel, T5Tokenizer, BitsAndBytesConfig

from ltx_video.models.autoencoders.causal_video_autoencoder import CausalVideoAutoencoder
from ltx_video.models.transformers.symmetric_patchifier import SymmetricPatchifier
from ltx_video.models.transformers.transformer3d import Transformer3DModel
from ltx_video.schedulers.rf import RectifiedFlowScheduler
from ltx_video.pipelines.pipeline_ltx_video import LTXVideoPipeline

MODEL_REPO = "Lightricks/LTX-Video"
MODEL_FILE = "ltxv-2b-0.9.8-distilled.safetensors"
TEXT_REPO = "PixArt-alpha/PixArt-XL-2-1024-MS"

print("Downloading/checking LTX checkpoint…")
CKPT = hf_hub_download(repo_id=MODEL_REPO, filename=MODEL_FILE)

with safe_open(CKPT, framework="pt") as f:
    metadata = f.metadata() or {}
    checkpoint_config = json.loads(metadata.get("config", "{}"))
allowed_steps = checkpoint_config.get("allowed_inference_steps", None)

print("Loading transformer…")
transformer = Transformer3DModel.from_pretrained(CKPT).to(device="cuda", dtype=DTYPE).eval()

print("Loading VAE…")
vae = CausalVideoAutoencoder.from_pretrained(CKPT).to(device="cuda", dtype=DTYPE).eval()
if hasattr(vae, "enable_tiling"):
    vae.enable_tiling()

print("Loading scheduler…")
scheduler = RectifiedFlowScheduler.from_pretrained(CKPT)

print("Loading tokenizer…")
tokenizer = T5Tokenizer.from_pretrained(TEXT_REPO, subfolder="tokenizer")

print("Loading text encoder (8-bit when supported)…")
TEXT_ENCODER_8BIT = False
try:
    qconfig = BitsAndBytesConfig(load_in_8bit=True)
    text_encoder = T5EncoderModel.from_pretrained(
        TEXT_REPO,
        subfolder="text_encoder",
        quantization_config=qconfig,
        device_map={"": 0},
    ).eval()
    TEXT_ENCODER_8BIT = True
except Exception as e:
    print("8-bit text encoder unavailable; falling back to FP16/BF16:", type(e).__name__)
    text_encoder = T5EncoderModel.from_pretrained(
        TEXT_REPO,
        subfolder="text_encoder",
        torch_dtype=DTYPE,
    ).to("cuda").eval()

patchifier = SymmetricPatchifier(patch_size=1)

pipe = LTXVideoPipeline(
    transformer=transformer,
    patchifier=patchifier,
    text_encoder=text_encoder,
    tokenizer=tokenizer,
    scheduler=scheduler,
    vae=vae,
    prompt_enhancer_image_caption_model=None,
    prompt_enhancer_image_caption_processor=None,
    prompt_enhancer_llm_model=None,
    prompt_enhancer_llm_tokenizer=None,
    allowed_inference_steps=allowed_steps,
)

gc.collect()
torch.cuda.empty_cache()

print("✅ MODEL IS HOT.")
print("Text encoder 8-bit:", TEXT_ENCODER_8BIT)
print(f"Allocated VRAM: {torch.cuda.memory_allocated()/1024**3:.2f} GB")
print("Do NOT rerun this cell between videos.")


In [ ]:
# 4) FAST GENERATOR ENGINE
import os, time, random, gc
from pathlib import Path

import numpy as np
import torch
import imageio
from PIL import Image

from ltx_video.inference import calculate_padding, prepare_conditioning
from ltx_video.utils.skip_layer_strategy import SkipLayerStrategy

OUTPUT_DIR = Path("/kaggle/working/lily_videos")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Official 0.9.8 2B-distilled first-pass schedule.
FAST_TIMESTEPS = [1.0000, 0.9937, 0.9875, 0.9812, 0.9750, 0.9094, 0.7250]

# Aggressive 4-step subset for the fastest previews.
TURBO_TIMESTEPS = [1.0000, 0.9812, 0.9094, 0.7250]

PRESETS = {
    "⚡ TURBO — 512×320 / 4 steps": (512, 320, TURBO_TIMESTEPS),
    "🔥 FAST — 640×384 / 7 steps": (640, 384, FAST_TIMESTEPS),
    "✨ BETTER — 704×416 / 7 steps": (704, 416, FAST_TIMESTEPS),
}

FPS = 15
DURATION_FRAMES = {
    "5 seconds": 73,    # N*8+1, ≈4.87 sec @ 15fps
    "10 seconds": 153,  # N*8+1, ≈10.2 sec @ 15fps
}

NEGATIVE = "worst quality, inconsistent motion, blurry, jittery, distorted, warped"

def _save_mp4(images, out_path, fps=FPS):
    # Official pipeline returns B,C,F,H,W in approximately [0,1].
    vid = images[0].permute(1, 2, 3, 0).detach().float().cpu().numpy()
    vid = np.clip(vid * 255.0, 0, 255).astype(np.uint8)

    writer = imageio.get_writer(
        str(out_path),
        fps=fps,
        codec="libx264",
        ffmpeg_params=["-pix_fmt", "yuv420p", "-movflags", "+faststart"],
        macro_block_size=None,
    )
    for frame in vid:
        writer.append_data(frame)
    writer.close()

def generate_video(prompt, image_path=None, preset="🔥 FAST — 640×384 / 7 steps",
                   duration="5 seconds", seed=0):
    if not prompt or not str(prompt).strip():
        raise ValueError("Write a prompt first.")

    width, height, timesteps = PRESETS[preset]
    num_frames = DURATION_FRAMES[duration]
    seed = int(seed) if seed not in (None, "") else 0
    if seed <= 0:
        seed = random.randint(1, 2_147_483_647)

    # Dimensions in presets are already divisible by 32.
    padding = calculate_padding(height, width, height, width)

    conditioning_items = None
    if image_path:
        conditioning_items = prepare_conditioning(
            conditioning_media_paths=[str(image_path)],
            conditioning_strengths=[1.0],
            conditioning_start_frames=[0],
            height=height,
            width=width,
            num_frames=num_frames,
            padding=padding,
            pipeline=pipe,
        )

    generator = torch.Generator(device="cuda").manual_seed(seed)
    out_path = OUTPUT_DIR / f"lily_{int(time.time())}_{seed}.mp4"

    torch.cuda.empty_cache()
    start = time.perf_counter()

    with torch.inference_mode():
        result = pipe(
            height=height,
            width=width,
            num_frames=num_frames,
            frame_rate=FPS,
            prompt=str(prompt).strip(),
            negative_prompt=NEGATIVE,

            # Distilled = CFG/STG not needed.
            timesteps=timesteps,
            guidance_scale=1.0,
            stg_scale=0.0,
            rescaling_scale=1.0,
            skip_block_list=[42],
            skip_layer_strategy=SkipLayerStrategy.AttentionValues,

            generator=generator,
            output_type="pt",
            conditioning_items=conditioning_items,

            # Official 0.9.8 distilled decode settings.
            decode_timestep=0.05,
            decode_noise_scale=0.025,
            stochastic_sampling=False,

            is_video=True,
            vae_per_channel_normalize=True,
            image_cond_noise_scale=0.15,
            mixed_precision=False,
            offload_to_cpu=False,
            device="cuda",
            enhance_prompt=False,
        )

    seconds = time.perf_counter() - start
    images = result.images
    _save_mp4(images, out_path, FPS)

    del result, images, conditioning_items
    gc.collect()
    torch.cuda.empty_cache()

    status = (
        f"✅ {duration} • {width}×{height} • {len(timesteps)} denoise steps • "
        f"seed {seed} • render {seconds:.1f}s • {GPU_NAME}"
    )
    return str(out_path), status

print("✅ Engine ready.")


In [ ]:
# 5) PHONE-FRIENDLY VIDEO STUDIO
#
# Run this cell and open the public Gradio link printed below on your iPhone.

import gradio as gr

def ui_generate(prompt, image, preset, duration, seed):
    try:
        image_path = image if image else None
        return generate_video(
            prompt=prompt,
            image_path=image_path,
            preset=preset,
            duration=duration,
            seed=seed,
        )
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        raise gr.Error("GPU ran out of VRAM. Switch to ⚡ TURBO 512×320 and try again.")
    except Exception as e:
        raise gr.Error(f"{type(e).__name__}: {e}")

with gr.Blocks(title="Lily Fast Video Studio") as demo:
    gr.Markdown(
        "## ⚡ Lily Fast Video Studio\n"
        "**Upload a first frame or leave it empty for text-to-video.** "
        "FAST is the default sweet spot; TURBO is for maximum speed."
    )

    with gr.Row():
        with gr.Column():
            image = gr.Image(
                type="filepath",
                label="Reference image (optional)",
                sources=["upload"],
            )
            prompt = gr.Textbox(
                label="What happens?",
                placeholder="She slowly turns toward the camera as her hair moves softly in the wind…",
                lines=4,
            )
            preset = gr.Radio(
                choices=list(PRESETS.keys()),
                value="🔥 FAST — 640×384 / 7 steps",
                label="Speed / quality",
            )
            duration = gr.Radio(
                choices=["5 seconds", "10 seconds"],
                value="5 seconds",
                label="Length",
            )
            seed = gr.Number(
                value=0,
                precision=0,
                label="Seed (0 = random)",
            )
            go = gr.Button("GENERATE ⚡", variant="primary")

        with gr.Column():
            video = gr.Video(label="Result", autoplay=True)
            status = gr.Markdown("Model is loaded and ready.")

    go.click(
        fn=ui_generate,
        inputs=[prompt, image, preset, duration, seed],
        outputs=[video, status],
    )

demo.queue(default_concurrency_limit=1)
demo.launch(share=True, debug=True)


### Which mode should you use?

- **⚡ TURBO 512×320 / 4 steps** — quickest possible previews. Use this while experimenting with prompts.
- **🔥 FAST 640×384 / 7 steps** — the default; best speed/quality compromise.
- **✨ BETTER 704×416 / 7 steps** — use only when the extra detail is worth the additional render time.

The first generation after model load can be a little slower because CUDA kernels are warming up. Later generations reuse the same loaded model.

**Important:** Kaggle's two-T4 option does not automatically make one video twice as fast. This notebook deliberately keeps a single generation on GPU 0 to avoid multi-GPU communication overhead.